In [1]:
import pennylane as qml
from pennylane import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
import os
import math


# Number of qubits
num_qubits = 5

# Initialize the device
dev = qml.device("lightning.qubit", wires=num_qubits)

In [2]:
# Construct the Hamiltonian terms
Hamiltonian_terms = []

# Interaction terms: XiX(i+1) + YiY(i+1) + ZiZ(i+1)
for i in range(num_qubits):
    Hamiltonian_terms.append(1.0 * (qml.PauliX(i) @ qml.PauliX((i+1)%num_qubits)) +
                                    (qml.PauliY(i) @ qml.PauliY((i+1)%num_qubits)) 
                                    + (qml.PauliZ(i) @ qml.PauliZ((i+1)%num_qubits)))

# Magnetic field terms: hZi
for i in range(num_qubits):
    Hamiltonian_terms.append(1.0 * qml.PauliZ(i))

# Define the Hamiltonian
Hamiltonian_operator = qml.Hamiltonian(coeffs=[1] * len(Hamiltonian_terms), observables=Hamiltonian_terms)

In [3]:

# Original rotosolve
gates = [qml.RX, qml.RY, qml.RZ]

def entangling_layer(num_qubits):
    # create ladder controlled-Z layer
    m=0
    n=1
    while m+1 < num_qubits:
        qml.CZ(wires=[m,m+1])
        m+=2

    while n+1 < num_qubits:
        qml.CZ(wires=[n,n+1])
        n+=2

# Define the quantum circuit
def circuit_roto(params, layers):

    for j in range(layers):
        # Assign all single qubit gates and thetas for all layers

        for k in range(num_qubits):
            qml.RX(params[k + 2 * num_qubits * j], wires = k)

        for n in range(num_qubits):
            qml.RY(params[(n + num_qubits) + 2 * num_qubits * j], wires = n)    

        
        entangling_layer(num_qubits)


# Define the cost function
@qml.qnode(dev)
def cost_roto(params, layers):
    circuit_roto(params, layers)
    return qml.expval(Hamiltonian_operator)


def opt_theta(d, params, cost, layers):
    params[d] = 0.0
    M_0 = cost(params,  layers)
    params[d] = np.pi / 2.0
    M_0_plus = cost(params,  layers)
    params[d] = -np.pi / 2.0
    M_0_minus = cost(params,  layers)
    a = np.arctan2(
        2.0 * M_0 - M_0_plus - M_0_minus, M_0_plus - M_0_minus
    )  # returns value in (-pi,pi]
    params[d] = -np.pi / 2.0 - a
    # restrict output to lie in (-pi,pi], a convention
    # consistent with the Rotosolve paper
    if params[d] <= -np.pi:
        params[d] += 2 * np.pi

# one cycle of rotosolve
def rotosolve_cycle(cost, params, layers, energy_vals):
    for d in range(len(params)):
        energy_vals.append(cost(params,  layers))
        opt_theta(d, params, cost, layers)

    return params, energy_vals


In [5]:

layers = [1]
n_steps = 10
trials = 20


for layer in layers:
    print("layers: ", layer)
    for j in range(trials):
        print("trial: ", j+1)
        params_rsol = np.random.uniform(-np.pi, np.pi, 2*num_qubits*layer, requires_grad=True)
        
        costs_rotosolve = []
        
        energy_vals = []

        for i in range(n_steps):
            print("iter", i+1)

            params_rsol, energy_vals = rotosolve_cycle(cost_roto, params_rsol,  layer, energy_vals)


        data_file = f"1DHeisenberg_{num_qubits}Q_rotosolve_{n_steps}cycles_{layer}layers_{trials}trials_Evals_A.xlsx"

        
        if not os.path.exists(data_file):
            df = pd.DataFrame()
            df.to_excel(data_file)

        df = pd.read_excel(data_file)
        
        if len(df.columns) < trials:
            df[f"col{len(df.columns)}"] = pd.Series(energy_vals)
            df.to_excel(data_file,index = False)
            
        else:
            break
        
    

layers:  1
trial:  1
iter 1
iter 2
iter 3
iter 4
iter 5
iter 6
iter 7
iter 8
iter 9
iter 10
trial:  2
iter 1
iter 2
iter 3
iter 4
iter 5
iter 6
iter 7
iter 8
iter 9
iter 10
trial:  3
iter 1
iter 2
iter 3
iter 4
iter 5
iter 6
iter 7
iter 8
iter 9
iter 10
trial:  4
iter 1
iter 2
iter 3
iter 4
iter 5
iter 6
iter 7
iter 8
iter 9
iter 10
trial:  5
iter 1
iter 2
iter 3
iter 4
iter 5
iter 6
iter 7
iter 8
iter 9
iter 10
trial:  6
iter 1
iter 2
iter 3
iter 4
iter 5
iter 6
iter 7
iter 8
iter 9
iter 10
trial:  7
iter 1
iter 2
iter 3
iter 4
iter 5
iter 6
iter 7
iter 8
iter 9
iter 10
trial:  8
iter 1
iter 2
iter 3
iter 4
iter 5
iter 6
iter 7
iter 8
iter 9
iter 10


KeyboardInterrupt: 